[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/03-OCRfacturas.ipynb)


# 03 — Comparando métodos de OCR: de lo clásico a los modelos que "ven"

En **`02-PDF_reporte`** vimos que `pypdf` solo sirve si el PDF ya trae texto seleccionable. En cuanto el "documento" es en realidad una foto, `extract_text()` regresa (casi) nada — la información sigue ahí, pero como **píxeles**, no como caracteres.

Aquí comparamos **cuatro formas distintas** de resolver eso sobre el mismo problema real: **extraer los datos de una factura** (número, fecha, proveedor, total) a partir de su imagen.

| Ronda | Método | Qué es |
|---|---|---|
| 1 | **Tesseract** | El motor de OCR "clásico", sin deep learning. Gratis, instantáneo, sin GPU. |
| 2 | **EasyOCR** | OCR con redes neuronales, entrenado para texto "del mundo real" (facturas, tickets, letreros). |
| 3 | **PaddleOCR** | OCR moderno, más robusto con layouts distintos. |
| 4 | **Un modelo de Hugging Face que "lee y entiende" (VLM)** | En vez de sacar texto plano, le *preguntas* directo: "dame el número de factura y el total, en JSON". |

Corremos las cuatro rondas sobre **las mismas 3 facturas** y comparamos contra el dato real — lo sabemos porque son parte de un dataset de facturas sintéticas con respuesta conocida. Al final, con la Ronda 4, probamos algo más: **¿importa cómo capturaste el documento?** (PDF renderizado limpio vs. una foto del PDF).


## Los datos

Usamos 3 facturas del dataset de Kaggle [**High-Quality Invoice Images for OCR**](https://www.kaggle.com/datasets/osamahosamabdellatif/high-quality-invoice-images-for-ocr) — no vienen en este repo (son de un tercero), así que hay que subirlas.

Sube estos 3 archivos cuando se te pida (o ya deberías tener `batch1-0081.jpg` de **`02-PDF_reporte`**):

- `batch1-0081.jpg` — factura sencilla, 1 producto.
- `batch1-0039.jpg` — factura media, 4 productos.
- `batch1-0001.jpg` — factura con tabla larga, 7 productos.

Si ya tienes la carpeta `batch_1/batch1_1/` en local, el notebook las toma de ahí directo, sin pedir nada.


In [ ]:
import importlib.util
from pathlib import Path

EN_COLAB = importlib.util.find_spec("google.colab") is not None

NOMBRES_FACTURAS = ["batch1-0081.jpg", "batch1-0039.jpg", "batch1-0001.jpg"]
CARPETA_LOCAL = Path("batch_1/batch1_1")


def localizar_facturas():
    rutas = {}
    for nombre in NOMBRES_FACTURAS:
        for candidato in (Path(nombre), CARPETA_LOCAL / nombre):
            if candidato.exists():
                rutas[nombre] = candidato
                break
    return rutas


rutas_facturas = localizar_facturas()

if EN_COLAB and len(rutas_facturas) < len(NOMBRES_FACTURAS):
    from google.colab import files

    faltantes = [n for n in NOMBRES_FACTURAS if n not in rutas_facturas]
    print("Sube estos archivos:", faltantes)
    files.upload()
    rutas_facturas = localizar_facturas()

assert len(rutas_facturas) == len(NOMBRES_FACTURAS), (
    f"Faltan facturas: {set(NOMBRES_FACTURAS) - set(rutas_facturas)}"
)
rutas_facturas


## El dato real

Como es un dataset con respuesta conocida, guardamos aquí los 4 campos correctos de cada factura. Así, en vez de "eyeballear" si cada método funcionó, lo comparamos automáticamente al final.


In [ ]:
DATO_REAL = {
    "batch1-0081.jpg": {
        "invoice_number": "15288019",
        "invoice_date": "09/07/2014",
        "seller_name": "Fernandez Ltd",
        "total": "1.71",
    },
    "batch1-0039.jpg": {
        "invoice_number": "35593328",
        "invoice_date": "06/08/2011",
        "seller_name": "Cruz-Carter",
        "total": "3004.96",
    },
    "batch1-0001.jpg": {
        "invoice_number": "51109338",
        "invoice_date": "04/13/2013",
        "seller_name": "Andrews, Kirby and Valdez",
        "total": "6204.19",
    },
}


## Ronda 1 — Tesseract (el clásico)

**Tesseract** no usa deep learning: detecta formas de caracteres contra patrones. Es viejo, rapidísimo y no necesita GPU — buen piso de comparación.

En Colab se instala con `apt-get`. En Mac local: `brew install tesseract`. En Windows: instalador desde el repo oficial de Tesseract.


In [ ]:
if EN_COLAB:
    !apt-get -qq install -y tesseract-ocr > /dev/null
    %pip install -q pytesseract


In [ ]:
import time

import pytesseract
from PIL import Image


def ocr_tesseract(ruta):
    return pytesseract.image_to_string(Image.open(ruta))


resultados_tesseract = {}
for nombre, ruta in rutas_facturas.items():
    t0 = time.time()
    texto = ocr_tesseract(ruta)
    resultados_tesseract[nombre] = {"texto": texto, "segundos": time.time() - t0}

for nombre, r in resultados_tesseract.items():
    print(f"--- {nombre} ({r['segundos']:.2f}s) ---")
    print(r["texto"][:400])
    print()


## De texto plano a campos: una regex mínima

Las 3 facturas usan **la misma plantilla** (es un dataset sintético), así que dos patrones bastan para sacar el número, la fecha y el total de cualquier texto plano — sin entrenar nada.

Ojo: el total vive dentro de una **tabla** (`Total  $ neto  $ IVA  $ bruto`). El OCR plano no siempre respeta el orden de las columnas al leer una tabla, así que es normal que el total falle más seguido que el número o la fecha — sobre todo en la factura de 7 renglones. Ese es justo el problema que atacan las rondas 3 y 4.


In [ ]:
import re


def extraer_campos(texto):
    numero = re.search(r"Invoice no:?\s*(\d+)", texto)
    fecha = re.search(r"Date of issue:?\s*([\d/]+)", texto)
    vendedor = re.search(r"Seller:?\s*\n?\s*([A-Za-z,.&\- ]+)", texto)
    total = re.findall(r"\$\s*(\d[\d.,\s]*\d|\d)", texto)

    return {
        "invoice_number": numero.group(1) if numero else None,
        "invoice_date": fecha.group(1) if fecha else None,
        "seller_name": vendedor.group(1).strip() if vendedor else None,
        # el último "$" de la tabla suele ser el Gross worth == total
        "total": total[-1] if total else None,
    }


## Función para comparar contra el dato real

La reutilizamos para las 4 rondas.


In [ ]:
import pandas as pd


def normalizar(valor):
    if valor is None:
        return ""
    return re.sub(r"[^a-z0-9]", "", str(valor).lower())


def campo_correcto(extraido, real):
    return normalizar(real) != "" and normalizar(real) in normalizar(extraido)


def evaluar_metodo(nombre_metodo, resultados, extractor):
    """resultados: {nombre_factura: {'texto'|'json': ..., 'segundos': ...}}"""
    filas = []
    for nombre_factura, r in resultados.items():
        real = DATO_REAL[nombre_factura]
        campos = extractor(r)
        fila = {"metodo": nombre_metodo, "factura": nombre_factura, "segundos": round(r["segundos"], 2)}
        for campo in ("invoice_number", "invoice_date", "seller_name", "total"):
            fila[campo] = "✅" if campo_correcto(campos.get(campo), real[campo]) else "❌"
        filas.append(fila)
    return pd.DataFrame(filas)


In [ ]:
tabla_tesseract = evaluar_metodo(
    "Tesseract", resultados_tesseract, lambda r: extraer_campos(r["texto"])
)
tabla_tesseract


## Ronda 2 — EasyOCR (deep learning "de toda la vida")

Mismo procedimiento, mismas 3 facturas, mismo extractor de campos — solo cambia el motor de OCR.


In [ ]:
if EN_COLAB:
    %pip install -q easyocr


In [ ]:
import easyocr

lector_easyocr = easyocr.Reader(["en"], gpu=False)


def ocr_easyocr(ruta):
    lineas = lector_easyocr.readtext(str(ruta), detail=0)
    return "\n".join(lineas)


resultados_easyocr = {}
for nombre, ruta in rutas_facturas.items():
    t0 = time.time()
    texto = ocr_easyocr(ruta)
    resultados_easyocr[nombre] = {"texto": texto, "segundos": time.time() - t0}

tabla_easyocr = evaluar_metodo(
    "EasyOCR", resultados_easyocr, lambda r: extraer_campos(r["texto"])
)
tabla_easyocr


## Ronda 3 — PaddleOCR (más robusto con layouts distintos)

PaddleOCR es otro motor con deep learning, con mejor manejo de rotación y de layouts variados que EasyOCR. (También existe **PP-StructureV3**, que reconstruye tablas completas — vale la pena explorarlo si quieren ir más lejos, pero aquí nos quedamos con el mismo extractor de texto plano para comparar parejo.)


In [ ]:
if EN_COLAB:
    %pip install -q paddlepaddle paddleocr


In [ ]:
from paddleocr import PaddleOCR

lector_paddle = PaddleOCR(use_angle_cls=True, lang="en")


def ocr_paddle(ruta):
    resultado = lector_paddle.ocr(str(ruta), cls=True)
    lineas = [linea[1][0] for bloque in resultado for linea in bloque]
    return "\n".join(lineas)


resultados_paddle = {}
for nombre, ruta in rutas_facturas.items():
    t0 = time.time()
    texto = ocr_paddle(ruta)
    resultados_paddle[nombre] = {"texto": texto, "segundos": time.time() - t0}

tabla_paddle = evaluar_metodo(
    "PaddleOCR", resultados_paddle, lambda r: extraer_campos(r["texto"])
)
tabla_paddle


## Ronda 4 — un modelo de Hugging Face que "lee y entiende" (VLM)

Hasta aquí sacamos **texto plano** y lo peinamos con regex. Un **VLM** (*vision-language model*) hace algo distinto: le mandas la imagen **y una pregunta**, y regresa la respuesta ya redactada — nada de regex.

Usamos **[Moondream2](https://huggingface.co/vikhyatk/moondream2)**: un VLM chiquito (~1.9B parámetros) pensado para correr rápido incluso en CPU — ideal para probarlo en vivo sin esperar una descarga enorme ni depender de GPU.


In [ ]:
if EN_COLAB:
    %pip install -q transformers accelerate einops


In [ ]:
import json as json_lib

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from PIL import Image

MOONDREAM_ID = "vikhyatk/moondream2"

modelo_moondream = AutoModelForCausalLM.from_pretrained(
    MOONDREAM_ID, trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
tokenizer_moondream = AutoTokenizer.from_pretrained(MOONDREAM_ID)
if torch.cuda.is_available():
    modelo_moondream = modelo_moondream.to("cuda")


El modelo card de Moondream ha cambiado de API entre versiones (`answer_question` en versiones viejas, `.query()` en las recientes). Si la celda de abajo truena, revisa el ejemplo de uso actual en la página del modelo en Hugging Face.


In [ ]:
PROMPT_JSON = (
    "Read this invoice and answer ONLY with a JSON object with these exact keys: "
    "invoice_number, invoice_date, seller_name, total. No explanation, just the JSON."
)


def parsear_json_de_texto(texto):
    """Los VLM a veces envuelven el JSON en texto o markdown; intentamos rescatarlo."""
    match = re.search(r"\{.*\}", texto, re.DOTALL)
    if not match:
        return {}
    try:
        return json_lib.loads(match.group(0))
    except json_lib.JSONDecodeError:
        return {}


def preguntar_moondream(imagen_o_ruta, pregunta):
    imagen = imagen_o_ruta if isinstance(imagen_o_ruta, Image.Image) else Image.open(imagen_o_ruta)
    return modelo_moondream.query(imagen.convert("RGB"), pregunta)["answer"]


resultados_moondream = {}
for nombre, ruta in rutas_facturas.items():
    t0 = time.time()
    respuesta = preguntar_moondream(ruta, PROMPT_JSON)
    resultados_moondream[nombre] = {"texto": respuesta, "segundos": time.time() - t0}
    print(f"--- {nombre} ({resultados_moondream[nombre]['segundos']:.2f}s) ---")
    print(respuesta)
    print()


In [ ]:
tabla_moondream = evaluar_metodo(
    "Moondream2 (VLM)",
    resultados_moondream,
    lambda r: parsear_json_de_texto(r["texto"]),
)
tabla_moondream


### Opcional (necesita GPU) — un VLM más grande y más preciso: Qwen2.5-VL

Moondream es rápido, pero **Qwen2.5-VL-7B** entiende documentos y tablas bastante mejor — es de los mejores VLM abiertos para esto hoy. Corre en el **T4 gratis de Colab** en 4-bit (`Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`), pero la primera descarga tarda varios minutos (~15 GB de pesos). Sáltate esta celda si no tienes GPU o si vas corto de tiempo.


In [ ]:
if EN_COLAB:
    %pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils


In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

QWEN_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

config_4bit = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
modelo_qwen = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_ID, quantization_config=config_4bit, device_map="auto"
)
procesador_qwen = AutoProcessor.from_pretrained(QWEN_ID)


def preguntar_qwen(ruta_imagen, pregunta):
    mensajes = [{
        "role": "user",
        "content": [
            {"type": "image", "image": str(ruta_imagen)},
            {"type": "text", "text": pregunta},
        ],
    }]
    texto_entrada = procesador_qwen.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    imagenes, videos = process_vision_info(mensajes)
    entradas = procesador_qwen(text=[texto_entrada], images=imagenes, videos=videos, return_tensors="pt").to(modelo_qwen.device)

    salida = modelo_qwen.generate(**entradas, max_new_tokens=200)
    salida_nueva = salida[:, entradas.input_ids.shape[1]:]
    return procesador_qwen.batch_decode(salida_nueva, skip_special_tokens=True)[0]


resultados_qwen = {}
for nombre, ruta in rutas_facturas.items():
    t0 = time.time()
    respuesta = preguntar_qwen(ruta, PROMPT_JSON)
    resultados_qwen[nombre] = {"texto": respuesta, "segundos": time.time() - t0}
    print(nombre, "->", respuesta)

tabla_qwen = evaluar_metodo("Qwen2.5-VL (opcional)", resultados_qwen, lambda r: parsear_json_de_texto(r["texto"]))
tabla_qwen


### ¿Importa cómo capturaste el documento? PDF renderizado vs. foto del PDF

Todo lo anterior fue con **fotos ya digitales** (el dataset de facturas). Pero en la vida real, muchas veces el punto de partida es un **PDF** (como el de `02-PDF_reporte`) y alguien lo **imprime y le toma una foto con el celular** en vez de mandar el archivo digital.

Comparamos el mismo modelo (Moondream2) sobre **la misma página**, capturada de dos formas:

1. **PDF renderizado limpio** — convertimos la página del PDF a imagen directamente (como si lo abrieras en una pantalla).
2. **"Foto del PDF"** — simulamos una foto tomada de esa misma página ya impresa: ángulo, sombra/reflejo, menos resolución, compresión JPEG.


In [ ]:
if EN_COLAB:
    %pip install -q pypdfium2

NOMBRE_PDF_EY = "ey_100_casos_rentables_ia_2026.pdf"
ruta_pdf_ey = Path(NOMBRE_PDF_EY)

if EN_COLAB and not ruta_pdf_ey.exists():
    from google.colab import files

    print(f"Sube {NOMBRE_PDF_EY} (el mismo PDF de 02-PDF_reporte)")
    files.upload()


In [ ]:
import pypdfium2 as pdfium

pdf_ey = pdfium.PdfDocument(str(ruta_pdf_ey))
pagina_limpia = pdf_ey[0].render(scale=2).to_pil().convert("RGB")
pagina_limpia


In [ ]:
import numpy as np


def simular_foto(imagen, angulo=6, escala=0.5, calidad_jpeg=40):
    """Aproxima una foto de celular a un documento impreso: rotación, menos resolución,
    sombra/gradiente de luz y compresión JPEG agresiva."""
    from io import BytesIO

    img = imagen.rotate(angulo, expand=True, fillcolor=(255, 255, 255))
    nuevo_ancho = int(img.width * escala)
    nuevo_alto = int(img.height * escala)
    img = img.resize((nuevo_ancho, nuevo_alto))

    # Gradiente de sombra/reflejo de izquierda a derecha
    gradiente = np.tile(np.linspace(0.65, 1.0, nuevo_ancho), (nuevo_alto, 1))
    arreglo = np.array(img).astype(float)
    for canal in range(3):
        arreglo[:, :, canal] *= gradiente
    img = Image.fromarray(np.clip(arreglo, 0, 255).astype("uint8"))

    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=calidad_jpeg)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


pagina_foto = simular_foto(pagina_limpia)
pagina_foto


In [ ]:
PROMPT_PAGINA = "Transcribe the main title and the first sentence of body text visible on this page."

respuesta_limpia = preguntar_moondream(pagina_limpia, PROMPT_PAGINA)
respuesta_foto = preguntar_moondream(pagina_foto, PROMPT_PAGINA)

print("--- PDF renderizado limpio ---")
print(respuesta_limpia)
print()
print("--- Foto simulada del PDF ---")
print(respuesta_foto)


Si el modelo transcribió bien el título en ambos casos pero se equivocó o inventó texto en la versión "foto", esa es la lección: **el modelo importa, pero la calidad de la captura del documento importa tanto o más.** Es la misma razón por la que, en producción, casi siempre conviene pedir el PDF/archivo original en vez de aceptar una foto.


## Comparando todo

Juntamos las tablas de las 4 rondas (agrega `tabla_qwen` si corriste la celda opcional).


In [ ]:
tablas = [tabla_tesseract, tabla_easyocr, tabla_paddle, tabla_moondream]
if "tabla_qwen" in dir():
    tablas.append(tabla_qwen)

comparacion = pd.concat(tablas, ignore_index=True)
comparacion


In [ ]:
resumen = comparacion.groupby("metodo").agg(
    segundos_promedio=("segundos", "mean"),
    aciertos_numero=("invoice_number", lambda s: (s == "✅").sum()),
    aciertos_fecha=("invoice_date", lambda s: (s == "✅").sum()),
    aciertos_vendedor=("seller_name", lambda s: (s == "✅").sum()),
    aciertos_total=("total", lambda s: (s == "✅").sum()),
)
resumen


## Cierre

- **Tesseract**: gratis y al instante, pero solo texto crudo — tú armas el resto.
- **EasyOCR**: mejor con fotos "del mundo real", pero más lento.
- **PaddleOCR**: más robusto ante layouts distintos que EasyOCR, velocidad similar.
- **Un VLM (Moondream2, Qwen2.5-VL)**: le pides el campo directo — sin regex, sin postprocesamiento — a cambio de más cómputo y (con modelos chicos) respuestas menos consistentes en formato.
- **La captura del documento importa tanto como el modelo**: un PDF limpio y una foto borrosa del mismo PDF no le dan la misma información a ningún método.

En producción, muchas empresas ya resuelven esto con **servicios administrados** (Google Document AI, AWS Textract, Azure Document Intelligence) que combinan varias de estas ideas y ya vienen entrenados para facturas — la ventaja de hacerlo "a mano" aquí es entender **qué está pasando por dentro** antes de delegarlo a una caja negra.


## Bonus opcional — otro tipo de documento: una identificación

*(Sáltate esta sección si vas corto de tiempo — es solo para mostrar que la misma técnica generaliza a otro tipo de documento, no solo facturas.)*

Reutilizamos el modelo de Hugging Face ya cargado (Moondream2) sobre una credencial **de ejemplo, generada para fines didácticos** (`fakeine.jpeg`) — no es un documento real.


In [ ]:
NOMBRE_INE = "fakeine.jpeg"
ruta_ine = Path(NOMBRE_INE)

if EN_COLAB and not ruta_ine.exists():
    from google.colab import files

    print(f"Sube {NOMBRE_INE}")
    files.upload()


In [ ]:
PROMPT_INE = (
    "Read this ID card and answer ONLY with a JSON object with these exact keys: "
    "nombre, fecha_nacimiento. No explanation, just the JSON."
)

respuesta_ine = preguntar_moondream(ruta_ine, PROMPT_INE)
print(respuesta_ine)
print(parsear_json_de_texto(respuesta_ine))


Con EasyOCR (Ronda 2) también se puede, aunque aquí no hay una plantilla fija como en las facturas — tocaría escribir reglas específicas para una credencial. Ese es, otra vez, el punto a favor del VLM: **generaliza a un documento distinto sin escribir una sola regex nueva.**
